# 10 minutes to MAGL curves

A rating curve turns a stage reading into a discharge. This package fits those curves
for the MAGL sensor network and draws them on a map.

This notebook walks the interface up from its smallest piece to the two scripts that
use all of it, `traditional_map.py` and `hierarchical_map.py`. Run it in order; each
step is the previous step with one thing added.

Run it on the `rating_curves` conda environment. The cells that fit models call NUTS
and take a few minutes each; the cells that fetch data use the local cache after the
first run.

## The five nouns

The whole interface is five things. Learn these and the rest is arguments.

| | what it is |
| --- | --- |
| `Sample` | paired stage and discharge measurements, plus where they came from |
| `ModelEntry` | one model's identity - its key, its color, which backend fits it |
| `Fit` | what came of fitting one model to one sample, successful or not |
| `SiteRating` | one monitoring site: its sample, its fits, and where it is on the map |
| `build_map` | the interactive HTML map, which is the usual end product |

Fitting is always the same two lines, whatever you are fitting:

```python
fit = entry.fit(sample)
site.attach_fit(fit)
```

Everything below is about getting good samples into the first line and useful sites
out of the second.

## 1. One sample

A `Sample` is the unit of data. Get one from the network:

In [ ]:
from limnotech_rating_curves import Sample, SiteRating, gage_sample, sensor_sample
from limnotech_rating_curves.models import catalog

gage = gage_sample("04176356")        # USGS field gagings at a gage

len(gage), gage.site_id, gage.stage_label

Or build one from data you have. `Sample.of` takes a `Sample` (returned unchanged), a
DataFrame (it finds the stage and discharge columns), or two arrays:

In [ ]:
mine = Sample.of([1.2, 2.4, 3.1], [4.0, 22.5, 61.0],
                 site_id="my_site", source="field_notes")

len(mine), mine.sample_id

`len(sample)` is the measurement count, and that number decides almost everything that
follows.

Two attributes are worth knowing apart:

- `sample.site_id` is the bare place the stage came from - `"04176356"`, `"SBR-09_WL"`.
  It titles plots and names export files.
- `sample.sample_id` is `"<source>:<site>"` - `"magl:SBR-01"`, `"colocated:SBR-01"`.
  Sensor SBR-01 feeds two different ratings from one stage record, so the source is
  part of the key. This is what a run over many sites files things under.

## 2. One model

A model is named by a key. `catalog.select` turns keys into the entries that can fit
them:

In [ ]:
print([entry.key for entry in catalog.select(["power_law"])])
print([entry.key for entry in catalog.select(None)])     # the curated default set
print([entry.key for entry in catalog.select("all")])    # every model

Groups (`"all"`, `"bdrc"`, `"ratingcurve"`) expand to their members, and the last three
keys are the forms the MAGL field workbooks use, refitted here to the measurements
rather than read from the workbook's typed coefficients.

A `ModelEntry` carries a model's identity: its key, how it reads in a legend, its color
on the map, the fewest measurements it can be fitted on, and which backend to call with
which arguments. `power_law_2seg` is "the ratingcurve power law with `segments=2`" -
the entry is what knows that, so nothing downstream has to.

## 3. One fit

This is the first slow cell - it runs NUTS.

In [ ]:
entry = catalog.select(["power_law"])[0]
fit = entry.fit(gage, method="nuts", seed=0, with_log_likelihood=True)

fit.ok, fit.status, fit.reason, fit.n

A `Fit` that failed is still returned rather than dropped, so a model that could not be
fitted shows up in the results with a reason instead of silently vanishing. That is the
one thing to remember about the type: **a `Fit` records an attempt, which may not have
produced a curve.** Check `.ok` before using one.

When it did fit:

In [ ]:
display(fit.curve.head())        # stage_ft, discharge_cfs, median, lower, upper
display(fit.metrics)             # in-sample scores
display(fit.config)              # what was actually used: segments, sampler, zero flow

fit.rating.predict(3.0)          # the live model object, evaluated exactly

`fit.idata` holds the posterior as arviz `InferenceData`, and `fit.bayes` holds the
PSIS-LOO / WAIC scores once they have been computed (step 9).

Two arguments matter more than they look:

- `method="nuts"` is full MCMC; `"advi"` is the fast variational fit. Use NUTS for
  anything you will quote. ADVI's normal approximation is least trustworthy exactly
  where these records are shortest.
- `with_log_likelihood=True` is what makes the ELPD model comparison possible at all.
  Pass it if you intend to compare models.

For a quick look at one site with no site or map machinery, `compare` does steps 2 and
3 together. It fits every model, so it is slow:

```python
from limnotech_rating_curves import compare

ratings = compare(gage, models="all")
ratings.metrics
```

## 4. One site

A `SiteRating` is a sample plus everywhere it belongs: its identity, its coordinates,
its cluster, and the fits made at it.

In [ ]:
site = SiteRating(sample_id="usgs_gage:04176356", source="usgs_gage",
                  label="USGS 04176356", sample=gage, coords=(41.78, -83.66))

site.attach_fit(fit)

site.fits, site.successful_fits

`attach_fit` replaces any earlier fit with the same model key, so refitting re-reads
rather than duplicating. Fits accumulate across models, which is what lets the score
table compare them.

You will rarely build one by hand. For the MAGL network:

In [ ]:
from limnotech_rating_curves.data.magl import assemble_magl_sites

sites = assemble_magl_sites(sites=["SR"], sources=["magl", "usgs_gage"],
                            min_points=3, years=5)

[(s.sample_id, len(s.sample)) for s in sites]

- `sites=` takes selection tokens: a sensor (`"SBR-09"`), a co-located station
  (`"SR-08"`), a cluster (`"SR"`), or a gage id (`"04176356"`). `None` takes
  everything. Only what you select is fetched, so a targeted run stays fast.
- `sources=` restricts to `"magl"` (sensor stage vs MAGL flow-sheet discharge),
  `"colocated"` (sensor stage vs USGS gagings), `"usgs_gage"` (the gage on its own).
- `min_points=` is the fewest measurements before a sensor is included at all.
- `years=` is how far back a USGS gage's measurements are taken. A channel changes, so
  an old gaging describes a rating that no longer applies; a gage with nothing that
  recent is left out entirely.

The sites come back with samples and coordinates filled in and `fits` empty.

## 5. Every site, one at a time

This is the whole of `traditional_map.py`. Nothing new - just the loop:

In [ ]:
entries = catalog.select(["power_law"])

for site in sites:
    if len(site.sample) == 0:
        continue
    for entry in entries:
        # with_log_likelihood is what makes the ELPD comparison possible at all
        fit = entry.fit(site.sample, method="nuts", seed=0, with_log_likelihood=True)
        site.attach_fit(fit)
        print(f"{site.sample_id} / {entry.key}: n={fit.n}, {fit.status}")

The loop is written out rather than hidden in a helper on purpose: which models get
fitted where is the decision you are making, and it should be visible in your script.

In [ ]:
from limnotech_rating_curves.export.mapview import build_map

path = build_map(sites, output_html="traditional_map.html",
                 title="Independent fits. Hover a site to see its models.")
path

`build_map` writes one self-contained HTML file. Sites with no coordinates, or no
successful fit, are left off the map - they are still in your `sites` list. Pass
`exports=True` to embed each fit's manifest and posterior as a downloadable zip.

## 6. The problem this network has

A one-segment power law spends three parameters on the mean. So at a site with:

- **2 measurements** - it cannot be fitted at all,
- **3 measurements** - it fits exactly, with nothing left to estimate scatter from,
- **4 or more** - it fits, with something left over.

Most MAGL sensors have two or three. That is the whole reason for the second script:
independent fitting has nothing to say about them.

## 7. Every site, together

`fit_hierarchical` fits every site in one model. Each site's scatter is drawn from a
population estimated in the same fit, so sites with measurements set the population and
the population supplies the sites without.

Note `min_points=2`, which is the point of the whole exercise:

In [ ]:
from limnotech_rating_curves import fit_hierarchical

sites = assemble_magl_sites(sites=["SR"], sources=["magl", "usgs_gage"],
                            min_points=2, years=5)

joint = fit_hierarchical([site.sample for site in sites], reference=False)
for site in sites:
    site.attach_marginal_fit(joint)

print(joint.summary())

This is one posterior over every site, so `fit_hierarchical` returns a
`HierarchicalFit`, not a list of `Fit`s:

In [ ]:
display(joint.population)                  # mu_sigma per cluster, tau_sigma, gamma
display(joint.sigma())                     # each site's scatter, cluster, n

joint.converged, list(joint.results), joint.skipped

In [ ]:
joint.rating(sites[0].sample_id).predict(3.0)

`site.attach_marginal_fit(joint)` pulls this site's marginal out of the joint posterior
and puts it on `site.fits`, where the map and the score table read it like any other
model's. That is why the same `build_map` call works for both scripts:

In [ ]:
path = build_map(sites, output_html="hierarchical_map.html",
                 title="Hierarchical power law. Hover a site to see its fit.")
path

A two-measurement site is fitted here because its scatter comes from the population,
its exponent is bounded by a prior, and its zero flow is held at its estimate - the two
points only have to place the level.

Only scatter is pooled. Level and zero flow are properties of one place, and the
exponent is bounded rather than shared because sharing it would assert that two sites
have the same cross-section shape.

`reference=False` is because the gages are already in `sites` - they were assembled
alongside the sensors, over the same five years. Left at `True`, the model would add the co-located
gages itself.

Check `joint.converged` before quoting anything: it is `True` only with no divergences
and R-hat within tolerance.

## 8. Which one to run

Run both. They differ exactly where you would expect:

| | independent | hierarchical |
| --- | --- | --- |
| unit of work | one site, one model | every site at once |
| min measurements | 3 (4 to say anything about scatter) | 2 |
| several models compared | yes | no - it is one model |
| a short record | not fitted, or fitted with no scatter estimate | borrows scatter from its cluster |
| a long record | unaffected | essentially unaffected |
| cost | one fit per site per model | one joint fit |

The sites with two or three measurements are where the two maps part company. A site
with twenty gagings should look almost identical in both; if it does not, something is
wrong with the joint fit and `joint.converged` is the first thing to check.

## 9. Scoring and saving

Fitting is not scoring. To compare models properly:

In [ ]:
import pandas as pd

from limnotech_rating_curves.data import usgs
from limnotech_rating_curves.model_selection import elpd

rows = []
for site in sites:
    if site.gage:
        # the gage's published USGS rating, to compare the fits against
        site.reference = usgs.published_reference(site.gage, site.sample)
    for fit in site.fits:
        if fit.ok:
            fit.bayes = catalog.get(fit.key).bayes_metrics(fit)   # PSIS-LOO / WAIC
    rows.extend(elpd.comparison_rows(site))

pd.DataFrame(rows).head()      # one row per successful model per site

To keep the fits. Only successful fits are written; each saved rating is a manifest
plus the posterior. The manifest is what makes it loadable - a bare `.nc` records draws
over parameter names and nothing that says which site or which stage axis:

In [ ]:
from limnotech_rating_curves import load_ratings, save_site

for site in sites:
    save_site(site, "output/fitted_curves")

load_ratings("output/fitted_curves")     # keyed by (site_id, model_key)

## Cheat sheet

```python
# data
sample = gage_sample("04176356")                       # or sensor_sample("SBR-09")
sample = Sample.of(frame, site_id="x", source="y")     # your own data
sites  = assemble_magl_sites(sites=["SR"], min_points=3, years=5)

# models
entries = catalog.select(["power_law", "spline"])      # or "all", or None

# fitting
fit = entry.fit(sample, method="nuts", seed=0, with_log_likelihood=True)
site.attach_fit(fit)
fit.ok, fit.status, fit.reason, fit.n, fit.curve, fit.rating, fit.metrics

# fitting everything together instead
joint = fit_hierarchical([s.sample for s in sites], reference=False)
site.attach_marginal_fit(joint)
joint.summary(), joint.converged, joint.population, joint.sigma()

# scoring, mapping, saving
fit.bayes = catalog.get(fit.key).bayes_metrics(fit)
site.reference = usgs.published_reference(site.gage, site.sample)
pd.DataFrame(elpd.comparison_rows(site))
build_map(sites, output_html="map.html", exports=True)
save_site(site, "output/fitted_curves"); load_ratings("output/fitted_curves")
```

## Where to go next

- `traditional_map.py`, `hierarchical_map.py` - the two scripts this notebook builds
  toward, runnable as they are
- `getting_started.ipynb` - the same ground with plots
- `extras.ipynb` - zero flow, datums, convergence diagnostics
- `../hierarchical.md` - why the joint model is built the way it is